# Body DEV: diagnostyka TrOCR
Dodaj body-dev-input.zip (wersja body-dev-kaggle-v2) jako prywatny dataset do Kaggle Input. ZIP i automatycznie rozpakowane pliki sa obslugiwane. Wlacz GPU + Internet, potem Run All. Notebook i wyniki pozostaw prywatne. To diagnostyka, nie benchmark. Pobierz koncowy ZIP wynikow.

In [ ]:
%pip install -q transformers==4.57.6 jiwer==4.0.0 huggingface_hub==0.36.0 sentencepiece==0.2.1

In [ ]:
"""Diagnostic only: user-corrected draft references, not approved benchmark labels."""
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import base64
import gc
import hashlib
import importlib.metadata
import io
import json
import unicodedata
import zipfile

MODELS = {
    'microsoft/trocr-base-printed': '93450be3f1ed40a930690d951ef3932687cc1892',
    'PiotrSty/trocr-pl-mixed-v3': '85d0c91c26f8e088849096dded7c9ba10b4cd9c9',
}
FROZEN = {'NA2_FT', 'Nowiny_z_Rakuz_FT', 'Powodzenia_FT'}


def digest(data):
    return hashlib.sha256(data).hexdigest()


def pack(manifest, target):
    manifest, target = Path(manifest), Path(target)
    if target.exists():
        raise FileExistsError(target)
    source = [json.loads(s) for s in manifest.read_text(encoding='utf-8').splitlines() if s.strip()]
    rows, images = [], {}
    for i, row in enumerate(source):
        path = (manifest.parent / row['image']).resolve()
        if not path.is_relative_to(manifest.parent.resolve()):
            raise ValueError('Image outside draft directory')
        data = path.read_bytes()
        if digest(data) != row['sha256'] or row['eligible_for_evaluation'] is not False:
            raise ValueError('Expected checksum-verified draft')
        name = f'images/{i:04d}.png'
        images[name] = data
        rows.append({k: row[k] for k in ['id', 'text', 'original_text', 'collection', 'page_id',
                                        'sha256', 'source_review_decision', 'review_status', 'eligible_for_evaluation']})
        rows[-1]['image'] = name
    provenance = {'scope': 'diagnostic-only', 'source_manifest_sha256': digest(manifest.read_bytes()),
                  'reference_status': 'User-corrected draft; repeat review explicitly skipped.',
                  'sampling': '63 count-matched line proposals from 11 of 19 selected regions. Eight regions failed line-count matching. Not full-page evaluation.',
                  'limitations': 'Geometry and Unicode issues remain; no independent double review, near-duplicate audit or proof of upstream training separation.'}
    with zipfile.ZipFile(target, 'x', zipfile.ZIP_DEFLATED) as archive:
        archive.writestr('manifest.json', json.dumps(rows, ensure_ascii=False))
        archive.writestr('provenance.json', json.dumps(provenance))
        for name, data in images.items():
            archive.writestr(name, data)
    return digest(target.read_bytes())


def load_input(data, expected_sha256):
    if digest(data) != expected_sha256:
        raise ValueError('Input ZIP checksum mismatch')
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError('Duplicate ZIP members')
        if sum(i.file_size for i in archive.infolist()) > 50_000_000:
            raise ValueError('Input too large')
        for name in names:
            if PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts or '\\' in name or ':' in name:
                raise ValueError('Unsafe ZIP member')
        rows = json.loads(archive.read('manifest.json'))
        provenance = json.loads(archive.read('provenance.json'))
        if provenance['scope'] != 'diagnostic-only' or not rows or len(rows) > 200:
            raise ValueError('Invalid diagnostic input')
        if len({r['id'] for r in rows}) != len(rows):
            raise ValueError('Duplicate line ID')
        images = []
        for r in rows:
            if r['collection'] in FROZEN or r['eligible_for_evaluation'] is not False:
                raise ValueError('Expected non-test draft records')
            if r['source_review_decision'] not in {'verified', 'proposed', 'needs-review'}:
                raise ValueError('Missing source review decision')
            if not r['text'].strip():
                raise ValueError('Empty reference requires explicit policy')
            image = archive.read(r['image'])
            if digest(image) != r['sha256']:
                raise ValueError('Image checksum mismatch')
            images.append(image)
    return rows, images, provenance


def metrics(rows, predictions):
    from jiwer import cer, wer
    if [r['id'] for r in rows] != [p['id'] for p in predictions]:
        raise ValueError('Reference/prediction ID mismatch')
    normalize = lambda t: ' '.join(unicodedata.normalize('NFC', t).split())
    result = {}
    subsets = {'all_draft_lines': list(range(len(rows))),
               'without_needs_review': [i for i, r in enumerate(rows) if r['source_review_decision'] != 'needs-review']}
    for label, indices in subsets.items():
        if not indices:
            result[label] = {'lines': 0, 'cer': None, 'wer': None}
            continue
        refs = [normalize(rows[i]['text']) for i in indices]
        hyps = [normalize(predictions[i]['text']) for i in indices]
        result[label] = {'lines': len(indices), 'cer': cer(refs, hyps), 'wer': wer(refs, hyps),
                         'lowercase_cer_diagnostic': cer([s.lower() for s in refs], [s.lower() for s in hyps]),
                         'errors': sum(predictions[i]['status'] != 'ok' for i in indices),
                         'empty': sum(not s for s in hyps)}
    return result


def run(data, expected_sha256, runner_sha256=None):
    rows, image_bytes, provenance = load_input(data, expected_sha256)
    import torch
    from PIL import Image
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    if not torch.cuda.is_available():
        raise RuntimeError('Enable GPU and Internet in Kaggle')
    torch.manual_seed(0)
    output = Path('/kaggle/working') / ('body-dev-diagnostic-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir(parents=True)
    def save(name, value):
        (output / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    save('input-manifest.json', rows)
    save('provenance.json', provenance)
    report = {'scope': 'DRAFT diagnostic, not benchmark or SOTA evidence', 'input_zip_sha256': expected_sha256,
              'runner_sha256': runner_sha256,
              'models': MODELS, 'gpu': torch.cuda.get_device_name(0),
              'environment': {p: importlib.metadata.version(p) for p in ['torch', 'transformers', 'Pillow', 'jiwer', 'huggingface_hub']},
              'generation': {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 256, 'dtype': 'float32'},
              'normalization': 'NFC and whitespace; additional lowercase CER reported separately',
              'uncertain_ids': [r['id'] for r in rows if r['source_review_decision'] == 'needs-review'],
              'reference_flags': {r['id']: sum(c == '\ufffd' or unicodedata.category(c) == 'Co' for c in r['text']) for r in rows},
              'results': {}}
    for model_id, revision in MODELS.items():
        model = None
        predictions = []
        try:
            processor = TrOCRProcessor.from_pretrained(model_id, revision=revision, trust_remote_code=False)
            model = VisionEncoderDecoderModel.from_pretrained(model_id, revision=revision, trust_remote_code=False).float().cuda().eval()
            eos = model.generation_config.eos_token_id
            eos = set(eos if isinstance(eos, list) else [eos])
            for row, content in zip(rows, image_bytes):
                pred = {'id': row['id'], 'text': '', 'status': 'error'}
                try:
                    with Image.open(io.BytesIO(content)) as im:
                        pixels = processor(images=im.convert('RGB'), return_tensors='pt').pixel_values.cuda()
                    with torch.inference_mode():
                        ids = model.generate(pixels, do_sample=False, num_beams=1, max_new_tokens=256)[0].tolist()
                    pred.update(text=processor.batch_decode([ids], skip_special_tokens=True)[0], status='ok',
                                token_ids=ids, ended_with_eos=ids[-1] in eos,
                                possibly_truncated=len(ids) >= 257 and ids[-1] not in eos)
                except Exception as exc:
                    pred['error'] = type(exc).__name__ + ': ' + str(exc)
                predictions.append(pred)
        except Exception as exc:
            predictions = [{'id': r['id'], 'text': '', 'status': 'error', 'error': type(exc).__name__ + ': ' + str(exc)} for r in rows]
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
        save(model_id.replace('/', '--') + '.json', predictions)
        report['results'][model_id] = metrics(rows, predictions)
        save('report.json', report)
        print(model_id, json.dumps(report['results'][model_id], indent=2))
    save('checksums.json', {p.name: digest(p.read_bytes()) for p in output.iterdir() if p.is_file()})
    archive = output.with_suffix('.zip')
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as z:
        for path in output.iterdir():
            z.write(path, path.name)
    from IPython.display import HTML, display
    encoded = base64.b64encode(archive.read_bytes()).decode('ascii')
    display(HTML('<a download="' + archive.name + '" href="data:application/zip;base64,' + encoded + '">Pobierz ZIP wynikow</a>'))
    print('Output:', archive)


EXPECTED_FILES = {'manifest.json': '44ea45fe345337d8d04c3764e6278594e2745428daf9b0a9d4bef4e79c660526', 'provenance.json': 'ffc7d7ec13487031f18d541ae8c848a2ec9f60786203ebe117d53de5c6268f9d', 'images/0000.png': '559431680d58a92bb3dbd98735dccdd690be9fb40e7ba32ff29adea2d920d05f', 'images/0001.png': 'd1939b62c9c170a1a255047c4dae894dab1c92ade1351bdd36bbf759a1eb39ff', 'images/0002.png': 'cb72f49b2b797da506da622985ea42dc335705a85b5ced71927c2bc88dc21658', 'images/0003.png': '973da84632f430339f1ee5e0c6ffb697791dbd5457be2a5b118030ff38148dfe', 'images/0004.png': 'ee06096cfbedc1937ceaebe29620776d5349059218d32e79c153a25812f2a894', 'images/0005.png': '5f5d4d387c9d51a70c75734222c0c4d4701e3ee84dd04a84c036bd5373ced1fd', 'images/0006.png': 'fa551d5b1d85bf6935aeaf1386720f749b10954e4ef7fcf5e51761c334c55226', 'images/0007.png': '1d8719c531feea2598d1fa3009f7b2693de39b64a2faf3b6e43c90a499867e07', 'images/0008.png': '7e972874686563345add2e0dfea685aa50e1e6bfb0a4b56b71d5d684079948f9', 'images/0009.png': '5dd50b6b5524813ff60b651cbd3923798b75942655f416b1cec43144e4e0b9c7', 'images/0010.png': '071222145d28c1cfeae53e5a16ec59789a9a2e55ee41349292ec4dedb8f8a6f0', 'images/0011.png': 'b0d2b83c8b17f033770733fafeb55d56e951e8c38c944c2f3e6692292c26f030', 'images/0012.png': '9ddd09cfd3b91f697e957b2061573c60eb6a14fd991cb0cb61e44550a70f13fd', 'images/0013.png': '23982684d8441e0cd1b5156ad00cda588258625ab60c708ed92d7acdbea899ed', 'images/0014.png': 'fb7e521a09cd3f8dd441684fe311bded95f64ea6ce8c066ddb09a6498e40308d', 'images/0015.png': '2ee076f4725a4485ffe5cf3c4c60f9e0f9eaa475546c99c25bf881bab83617c5', 'images/0016.png': 'eb3c1b552533cd06ac06a061784fd430e6c33ce93ba27936f07c3400f011cac7', 'images/0017.png': '54eeb77f14a7351d06403176b3dff96afdc61165d8ed501bd3d5e4ea51fa39f5', 'images/0018.png': 'd74f98c3392d2f7d450a692a6aa4b47ecd809864adcfecf3c75039ab112712f3', 'images/0019.png': 'fd80462acd65d851129ff142f0873a630d2f631a3fa7aa1edbcffdc643b0abc0', 'images/0020.png': 'f34aa2ee0feb06585e05ecb4227b0fc460d4e447c7b18f8427b92795a2beec08', 'images/0021.png': 'f6aa213116c9d40e16a94e85afa25b027469bbac0bf9c35497b2ee691b0284ef', 'images/0022.png': '763ae8d0cce59b147bc46be93756a6bcab62a46e800b556ee2e97bba6a126d90', 'images/0023.png': '45b07bcaf02dc7feecb5e5f30e00c3f1ab9ef84f69341373bd7d6fa2ddda7e6f', 'images/0024.png': 'f26aabc5fc7ba1eaaa4d0490b534a12a8d9988ac35204cc0074bdfc197e085cf', 'images/0025.png': '659dd1ee4dec21f1e9b3981569db7c3ecbb49717bb0891a1998a9956402e8e31', 'images/0026.png': '597c91c3186bc6ca332607c551b9a70756478893dbd0f5a8fb5eaa2fd95c8dcb', 'images/0027.png': '5191871a77d44b4ff20782838b31b7f2af9c8380a086afe1597299d1f4d4beda', 'images/0028.png': '50d375773eb3df1698a83d3a06c9fd99ff8e94dce6ab0d48910e9d1e3bac5aa6', 'images/0029.png': '47caaefe04c0c2b50e38404bad6aaf1dee360b900b0cf668945952bdd12d481c', 'images/0030.png': 'c72d4685a13465c3a2335979210abfc426502f42894ecc2f511d565a4f0df528', 'images/0031.png': 'ac88217a5a48b7de10131bb08d1666b4ba3b42d93baf9fd9ebcfb928dc61ac11', 'images/0032.png': 'b1541703f77eecd357189fe7032935949bf8e26680a1941de20045105b4f444a', 'images/0033.png': 'e298ec570d6f6df749e086d382f6e90feae7ec508ae8f2ab37aee543719a7da8', 'images/0034.png': '167c64f6361dd67d8b5cd17c4a4d7927c9d02910edb56d71e67564320ebd0565', 'images/0035.png': '9534ebe7333cb819cd96b5cf0aa18cf153d025bf41fb9c83370e5ebea3c64537', 'images/0036.png': 'b962c306c49a9c09ce45be5aa9c2969444ee8f4e553a8974f0fe48fd8c855e07', 'images/0037.png': '45532f774d8cbe5c67885e6e740195cff3887c7a191ca16856df65d1ecaa473a', 'images/0038.png': '0c0184b44f7b656af1ccd34c5b35ce99b8b5dacc294524e190921752ccfed00b', 'images/0039.png': 'd3e0ac82f34e7306b72a8c053060ee52166f561dfe5fc469f8ee5e2f77683f9a', 'images/0040.png': '8fd1d94c25621cb903cb6d6f854a216da18211ce2168747282ba41839d123d20', 'images/0041.png': '47c77b2b069ec20cd7fd6289b1c9e4c60453c69aa17205ef7d5ae2fcfbc1f41e', 'images/0042.png': '5eaeb715245088d8b07b2557e28fdd5bfe9fddcb1f6631afa52ea6f2126d8791', 'images/0043.png': '5c7c73abf0bcfc361cffb23a275bf679dd309893f4c39fe6ee88daa659f7dbce', 'images/0044.png': 'e8eca47049dc18c6756d8364f1296e6c7d728da0f6ec6dcba4d2116cb26d61fa', 'images/0045.png': '8b8468e0799425db3c9c9285475b8d8c29136b439ad5eea933704558bee96fa0', 'images/0046.png': 'a458eeb3eb78576d573f76239dff1ad91f90b757eb8852a76ec42f8b313dca4e', 'images/0047.png': '7e6e8d6b55aae47000d0c534edb1f77f537efe3f5b4fd6dd4c304d2f1872aac8', 'images/0048.png': '91976fb2148110b264432426c79b4095ee452ea284dd3dbab29097ea23d19af4', 'images/0049.png': '31a0f67711b4c0935f4c0374d8aba5909789d0f4e0462ef2fd5c09a2e41efd1c', 'images/0050.png': '7efe2a1b6fcc4cfa774254eb454afcf42124af3be45758b9d92411b63ac992e9', 'images/0051.png': '63c5ea86db84963f8652a02afbe7d51c41661c163f801423b20d806fad9899b3', 'images/0052.png': '29b54acdf8535413220dbba60669b48e496838893fa5348a4fed7a7cf7c47b8a', 'images/0053.png': '49c2988080199d9a8a21124c5de4acbca88068cbec2ccc8606e7ff8b621d6da4', 'images/0054.png': 'cea82248a44d339bf99d391babf6e865031fd2d3ddca91a7bc03127a7fad8718', 'images/0055.png': '40deb9bc664f944344cc900f255d7c49bc5b4964b899bc81066d55abac56bb42', 'images/0056.png': '48ae63110331c563d2208a6a1cea1eb49db14de8ef522804d6995e5ab5312ad0', 'images/0057.png': 'cd77315430df068e46de6d5f358e688e85d89227b5d0d23b0e55a9e22eb792c7', 'images/0058.png': 'ed9f19f83279c001c0715f9aacb17fa63afa023ffb3a223c7198b6a3e9cbe8ee', 'images/0059.png': 'e1bd329f863d0450cc8382ad3c7981fc68c054848f213c1d8dbe3941e3946341', 'images/0060.png': '36759219aeba5d7f877ce18663661d9e657d21117889a5cccfba33a25937d182', 'images/0061.png': 'afdf66ef84af3451c0add766419175f902bd06881bdb72ec3e7d91e4161ea619', 'images/0062.png': '665371b00be884f61873634b71db84931b6db17075f061e5e40e3f330a2e9626'}
ORIGINAL_ZIP_SHA256 = '913ded2bd5d742099567a2652460baaf3c4a8bb16625492d0931607396dbdd55'
RUNNER_SHA256 = 'ac6c6267a1fed5d60dfffb6697877acea9ab01171cae05a4bf2a8b1043888580'
matches = [p for p in Path("/kaggle/input").rglob("body-dev-input.zip") if digest(p.read_bytes()) == ORIGINAL_ZIP_SHA256]
if len(matches) == 1:
    payload = matches[0].read_bytes()
else:
    manifests = [p for p in Path("/kaggle/input").rglob("manifest.json") if digest(p.read_bytes()) == EXPECTED_FILES["manifest.json"]]
    if len(manifests) != 1:
        raise RuntimeError("Add the private body-dev-input.zip from body-dev-kaggle-v2 to Kaggle Input. ZIP or automatically extracted files are supported.")
    root = manifests[0].parent
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, "w", zipfile.ZIP_DEFLATED) as archive:
        for name, expected in EXPECTED_FILES.items():
            content = (root / name).read_bytes()
            if digest(content) != expected:
                raise ValueError("Input checksum mismatch: " + name)
            archive.writestr(name, content)
    payload = buffer.getvalue()
run(payload, digest(payload), RUNNER_SHA256)
